# Task 1 — Thematic Factor Discovery: Factor Extraction

Covers **Task 1, Sub-step 2**: use the LLM to extract thematic factors from subsection JSONs produced by Task 1.1.

For each filing:
1. **Route questions** — match relevant analytical questions to each subsection via keyword matching
2. **Chunk-level Q&A** — call the LLM on each subsection with its routed questions
3. **Synthesis** — consolidate multi-subsection answers for the same factor into one entry

| | |
|---|---|
| **Input** | `output/subsections/{TICKER}/{STEM}_subsections.json` |
| **Output** | `output/factors/{TICKER}/{STEM}_factors.json` |
| **Model** | `deepseek-ai/DeepSeek-R1-Distill-Qwen-14B` via vLLM on ACCRE |
| **GPU** | Required — RTX A6000, account `p_dsi_acc` |

## Step 1: Imports & Configuration

Set up paths, logging, and LLM constants. The vLLM server must be running before executing Step 3+.

In [ ]:
import json
import logging
import sys
import time
import threading
import re
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Optional, List

from pydantic import BaseModel
from openai import OpenAI
from tqdm.notebook import tqdm

# ── Paths ────────────────────────────────────────────────────────────────
_cwd = Path.cwd()

_search = _cwd
FILLINGS_ROOT = None
for _ in range(6):
    if (_search / "Data").is_dir():
        FILLINGS_ROOT = _search
        break
    _search = _search.parent

if FILLINGS_ROOT is None:
    raise FileNotFoundError(
        f"Could not find Data/ directory in any parent of {_cwd}. "
        "Make sure the notebook is inside the fillings/ project tree."
    )

SUBSECTIONS_DIR = _cwd / "output" / "subsections"
FACTORS_DIR = _cwd / "output" / "factors"
QUESTIONS_PATH = _cwd / "questions.json"
TICKER_MAPPING_PATH = _cwd / "ticker_mapping.json"

FACTORS_DIR.mkdir(parents=True, exist_ok=True)

# ── LLM Constants ────────────────────────────────────────────────────────
BASE_URL = "http://127.0.0.1:8000/v1"
API_KEY = "local"
MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
TEMPERATURE = 0.6              # DeepSeek-R1 recommends 0.6; temp=0 causes degenerate output
MAX_RETRIES = 5
RETRY_BASE_DELAY = 2.0
REQUEST_TIMEOUT = 600          # 10 min — reasoning models are slower
MAX_IN_FLIGHT_LLM = 4
BATCH_TOKEN_LIMIT = 6000       # max subsection tokens per batch
CHUNK_MAX_TOKENS = 16384       # reasoning models need headroom for <think> + answer
SYNTHESIS_MAX_TOKENS = 8192

# ── Logging ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("task_1_2")

# ── Pydantic Response Schemas ────────────────────────────────────────────

class ChunkAnswer(BaseModel):
    key: str
    found: bool = True
    summary: str = ""
    evidence: str = ""
    subsection: str = ""

class ChunkQAResponse(BaseModel):
    answers: List[ChunkAnswer]

class SynthesisResponse(BaseModel):
    summary: str
    evidence: List[str]

print(f"Fillings root:    {FILLINGS_ROOT}")
print(f"Subsections dir:  {SUBSECTIONS_DIR}")
print(f"Factors dir:      {FACTORS_DIR}")
print(f"Questions file:   {QUESTIONS_PATH} (exists={QUESTIONS_PATH.exists()})")
print(f"Ticker mapping:   {TICKER_MAPPING_PATH} (exists={TICKER_MAPPING_PATH.exists()})")
print(f"Model:            {MODEL}")
print(f"Temperature:      {TEMPERATURE}")
print(f"Timeout:          {REQUEST_TIMEOUT}s per request")
print(f"Concurrency:      {MAX_IN_FLIGHT_LLM} in-flight LLM calls")
print(f"Batch limit:      {BATCH_TOKEN_LIMIT} tokens per batch")
print(f"JSON enforcement: response_format=json_object + Pydantic validation")

## Step 2: LLM Client Utilities

OpenAI-compatible client with:
- **Concurrency control** — `BoundedSemaphore` limits concurrent LLM calls
- **Exponential backoff** — retries on transient failures
- **JSON parsing** — strips markdown code blocks, extracts JSON from mixed output

In [ ]:
# Global concurrency limiter
_semaphore = threading.BoundedSemaphore(MAX_IN_FLIGHT_LLM)


def call_llm(
    client: OpenAI,
    system_prompt: str,
    user_prompt: str,
    temperature: float = TEMPERATURE,
    max_tokens: int = CHUNK_MAX_TOKENS,
    force_json: bool = False,
) -> Optional[str]:
    """
    Call the LLM with retry logic and concurrency control.
    If force_json=True, adds response_format={"type": "json_object"} to force valid JSON.
    Returns the response content string, or None on failure.
    """
    kwargs: dict[str, Any] = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "timeout": REQUEST_TIMEOUT,
    }

    if force_json:
        kwargs["response_format"] = {"type": "json_object"}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with _semaphore:
                response = client.chat.completions.create(**kwargs)
            return response.choices[0].message.content

        except Exception as e:
            delay = RETRY_BASE_DELAY * (2 ** (attempt - 1))
            log.warning(
                f"  LLM call attempt {attempt}/{MAX_RETRIES} failed: {e}. "
                f"Retrying in {delay}s..."
            )
            if attempt < MAX_RETRIES:
                time.sleep(delay)
            else:
                log.error(f"  LLM call failed after {MAX_RETRIES} attempts: {e}")
                return None


def parse_json_response(content: str) -> Optional[Any]:
    """Parse JSON from LLM response, handling markdown code blocks and <think> tags."""
    content = content.strip()

    # Strip closed <think>...</think> blocks (DeepSeek-R1 reasoning output)
    content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()

    # Strip unclosed <think> blocks (truncated reasoning — no closing tag)
    if "<think>" in content:
        content = re.sub(r"<think>.*", "", content, flags=re.DOTALL).strip()

    # Strip markdown code blocks if present
    if content.startswith("```"):
        content = content.split("\n", 1)[1] if "\n" in content else content[3:]
        if content.endswith("```"):
            content = content[:-3]
        content = content.strip()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass

    # Fallback: find the first { or [ and last } or ]
    start_obj = content.find("{")
    start_arr = content.find("[")
    if start_obj == -1 and start_arr == -1:
        log.error(f"  No JSON found in response: {content[:200]}...")
        return None

    if start_arr != -1 and (start_obj == -1 or start_arr < start_obj):
        start = start_arr
        end = content.rfind("]")
    else:
        start = start_obj
        end = content.rfind("}")

    if end == -1 or end <= start:
        log.error(f"  Malformed JSON in response: {content[:200]}...")
        return None

    try:
        return json.loads(content[start:end + 1])
    except json.JSONDecodeError as e:
        log.error(f"  JSON parse error: {e}. Content: {content[start:start+200]}...")
        return None


def call_llm_json(
    client: OpenAI,
    system_prompt: str,
    user_prompt: str,
    temperature: float = TEMPERATURE,
    max_tokens: int = CHUNK_MAX_TOKENS,
    response_model: Optional[type[BaseModel]] = None,
) -> Optional[Any]:
    """
    Call the LLM and parse the response as JSON.
    Uses response_format=json_object to force valid JSON output.
    If response_model is provided, validates with Pydantic after parsing.
    """
    content = call_llm(
        client, system_prompt, user_prompt,
        temperature, max_tokens,
        force_json=True,
    )
    if content is None:
        return None

    parsed = parse_json_response(content)
    if parsed is None:
        return None

    # Validate with Pydantic if response_model is provided
    if response_model is not None:
        try:
            validated = response_model.model_validate(parsed)
            return validated.model_dump()
        except Exception as e:
            log.error(f"  Pydantic validation failed: {e}")
            return None

    return parsed


# ── Create client & health check ─────────────────────────────────────────
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

try:
    models = client.models.list()
    available = [m.id for m in models.data]
    print(f"vLLM is reachable. Available models: {available}")
    if MODEL not in available:
        print(f"WARNING: {MODEL} not in available models!")
except Exception as e:
    print(f"WARNING: vLLM health check failed: {e}")
    print("The LLM server must be running before executing Steps 4+.")

## Step 3: Question Routing

Maps subsection titles to relevant question subsets using keyword matching.
Reduces noise and improves LLM focus by only asking relevant questions per subsection.

- **14 keyword categories** covering revenue, costs, supply chain, capital, competition, labor, regulation, macro, outlook, technology, ESG, airlines, defense, and industrial
- **Fallback**: if routing yields < 10 questions, all applicable questions are sent
- **Sub-sector filtering**: `ticker_mapping.json` maps tickers to sub-sectors; sub-sector-specific questions only apply to matching tickers

In [ ]:
# ── Keyword -> category mapping ─────────────────────────────────────────

SUBSECTION_KEYWORDS: dict[str, list[str]] = {
    "demand_revenue": [
        "revenue", "sales", "demand", "booking", "backlog", "order",
        "pricing", "passenger", "yield", "volume", "growth", "customer",
        "operating revenue", "net revenue", "commercial",
    ],
    "cost_margins": [
        "cost", "expense", "margin", "operating expense", "cogs",
        "salaries", "wages", "fuel", "material", "maintenance",
        "restructuring", "efficiency", "operating income", "profitability",
    ],
    "supply_chain_operations": [
        "supply chain", "manufacturing", "production", "capacity",
        "inventory", "logistics", "operations", "quality", "safety",
        "facilities", "utilization",
    ],
    "capital_allocation": [
        "liquidity", "capital", "debt", "cash flow", "financing",
        "credit", "borrowing", "dividend", "buyback", "repurchase",
        "balance sheet", "leverage", "investment", "capex",
        "acquisition", "divestiture",
    ],
    "competitive_position": [
        "competition", "competitive", "market share", "market position",
        "differentiation", "customer",
    ],
    "labor_workforce": [
        "employee", "workforce", "labor", "union", "headcount",
        "talent", "hiring", "personnel", "collective bargaining",
        "pilot", "strike",
    ],
    "regulatory_legal": [
        "regulation", "regulatory", "legal", "litigation",
        "government", "compliance", "environmental", "emission",
        "law", "legislation", "antitrust", "faa", "dot",
    ],
    "macro_external": [
        "economic", "economy", "inflation", "interest rate",
        "foreign exchange", "currency", "geopolitical", "macro",
        "recession", "gdp",
    ],
    "outlook_guidance": [
        "outlook", "guidance", "forward", "future", "expect",
        "forecast", "strategy", "strategic", "plan", "risk",
        "uncertainty", "trend", "catalyst",
    ],
    "technology_innovation": [
        "technology", "innovation", "digital", "automation",
        "r&d", "research", "development", "new product", "launch",
        "modernization",
    ],
    "esg_sustainability": [
        "esg", "sustainability", "climate", "environmental",
        "carbon", "emission", "renewable", "governance",
    ],
    "airlines_transport": [
        "fuel", "load factor", "asm", "rpm", "rasm", "casm",
        "fleet", "aircraft", "route", "passenger", "cargo",
        "seat mile", "travel", "airline",
    ],
    "defense_government": [
        "defense", "military", "contract", "backlog", "program",
        "classified", "government", "dod", "pentagon", "f-35",
        "missile", "fighter", "weapon", "radar", "satellite",
        "international sales", "fms",
    ],
    "industrial_infrastructure": [
        "construction", "infrastructure", "equipment", "dealer",
        "aftermarket", "service revenue", "replacement cycle",
        "fleet age", "distributor", "rental",
    ],
}

FALLBACK_CATEGORIES = [
    "demand_revenue",
    "cost_margins",
    "outlook_guidance",
]

# ── Caches ────────────────────────────────────────────────────────────────
_questions_cache: Optional[dict] = None
_ticker_mapping_cache: Optional[dict] = None


def load_questions() -> dict:
    """Load the questions taxonomy (cached)."""
    global _questions_cache
    if _questions_cache is None:
        with open(QUESTIONS_PATH) as f:
            _questions_cache = json.load(f)
    return _questions_cache


def load_ticker_mapping() -> dict:
    """Load the ticker-to-subsector mapping (cached)."""
    global _ticker_mapping_cache
    if _ticker_mapping_cache is None:
        with open(TICKER_MAPPING_PATH) as f:
            _ticker_mapping_cache = json.load(f)
    return _ticker_mapping_cache


def load_ticker_subsector(ticker: str) -> str:
    """Get the sub-sector for a ticker."""
    mapping = load_ticker_mapping()
    for sector, tickers in mapping.items():
        if ticker in tickers:
            return sector
    return "general"


def get_applicable_questions(ticker: str) -> dict[str, dict]:
    """
    Get all questions applicable to a ticker based on its sub-sector.
    Returns: {category_name: {key: question_text, ...}, ...}
    """
    questions = load_questions()
    subsector = load_ticker_subsector(ticker)

    applicable = {}
    for category_name, category_data in questions.items():
        cat_subsector = category_data["subsector"]
        if cat_subsector == "universal" or cat_subsector == subsector:
            applicable[category_name] = category_data["questions"]

    return applicable


def route_questions(
    subsection_name: str,
    subsection_text: str,
    all_questions: dict[str, dict],
) -> list[dict]:
    """
    Route relevant questions to a subsection based on keyword matching.

    Returns list of {key, category, question} dicts for questions to ask.
    """
    # Combine subsection name and beginning of text for matching
    search_text = (subsection_name + " " + subsection_text[:500]).lower()

    # Score each category by keyword matches
    category_scores: dict[str, int] = {}
    for category, keywords in SUBSECTION_KEYWORDS.items():
        score = 0
        for kw in keywords:
            if kw.lower() in search_text:
                score += 1
        if score > 0:
            category_scores[category] = score

    # Select categories with matches
    matched_categories = set(category_scores.keys())

    # Always include fallback categories
    matched_categories.update(FALLBACK_CATEGORIES)

    # For risk factor subsections, also include regulatory and outlook
    if "risk" in subsection_name.lower():
        matched_categories.update(["regulatory_legal", "outlook_guidance", "macro_external"])

    # Build the question list from matched categories
    routed: list[dict] = []
    for category_name, questions in all_questions.items():
        if category_name in matched_categories:
            for key, question_text in questions.items():
                routed.append({
                    "key": key,
                    "category": category_name,
                    "question": question_text,
                })

    # If routing produced very few questions (< 10), add all questions
    if len(routed) < 10:
        routed = []
        for category_name, questions in all_questions.items():
            for key, question_text in questions.items():
                routed.append({
                    "key": key,
                    "category": category_name,
                    "question": question_text,
                })

    return routed


# ── Validation ────────────────────────────────────────────────────────────
questions = load_questions()
total_q = sum(len(cat["questions"]) for cat in questions.values())
print(f"Loaded {total_q} questions across {len(questions)} categories")

mapping = load_ticker_mapping()
total_tickers = sum(len(v) for v in mapping.values())
print(f"Loaded {total_tickers} tickers across {len(mapping)} sub-sectors")

# Test routing on a sample ticker
test_ticker = "AAL"
test_q = get_applicable_questions(test_ticker)
test_total = sum(len(v) for v in test_q.values())
print(f"\n{test_ticker} sub-sector: {load_ticker_subsector(test_ticker)}")
print(f"{test_ticker} applicable questions: {test_total} across {len(test_q)} categories")
for cat, qs in test_q.items():
    print(f"  {cat}: {len(qs)} questions")

## Step 4: Chunk-Level Q&A (Phase 1)

Subsections are **batched** (~6000 tokens per batch) to reduce total LLM calls by ~5-8x. Each batch is sent as a single prompt with all subsections clearly labeled, and the LLM tags each answer with its source subsection.

Batches are processed **concurrently** (up to 8 in-flight) via ThreadPoolExecutor, leveraging vLLM's continuous batching on the GPU.

In [ ]:
CHUNK_SYSTEM_PROMPT = """\
You are a financial analyst extracting thematic factors from SEC filing text. Respond with valid JSON matching the required schema."""


def batch_subsections(
    subsections: list[dict], max_tokens: int = BATCH_TOKEN_LIMIT
) -> list[list[dict]]:
    """Group subsections into batches that fit within a token budget."""
    batches: list[list[dict]] = []
    current_batch: list[dict] = []
    current_tokens = 0

    for sub in subsections:
        tokens = sub.get("token_estimate", len(sub["text"]) // 4)
        if current_batch and current_tokens + tokens > max_tokens:
            batches.append(current_batch)
            current_batch = [sub]
            current_tokens = tokens
        else:
            current_batch.append(sub)
            current_tokens += tokens

    if current_batch:
        batches.append(current_batch)
    return batches


def build_batch_prompt(batch: list[dict], questions: list[dict]) -> str:
    """Build the user prompt for a batch of subsections."""
    subsection_blocks = []
    for i, sub in enumerate(batch, 1):
        subsection_blocks.append(
            f"## Subsection {i}: {sub['name']}\n"
            f"(Section: {sub['section']})\n\n"
            f"### Text:\n{sub['text']}"
        )

    question_block = "\n".join(
        f"- [{q['key']}] {q['question']}"
        for q in questions
    )

    subsections_text = "\n\n---\n\n".join(subsection_blocks)

    return f"""{subsections_text}

---

### Questions:
{question_block}

For each question where the text contains relevant information, include it in the "answers" array with:
- "key": the question key
- "found": true
- "summary": 1-3 sentence directional answer (improving, declining, stable, etc.)
- "evidence": verbatim quote from the text (max 2 sentences)
- "subsection": exact subsection name where you found the information

Only include questions where the text clearly addresses them. If none are answered, return {{"answers": []}}."""


def process_batch(
    client: OpenAI, batch: list[dict], questions: list[dict]
) -> list[dict]:
    """
    Run chunk-level Q&A on a batch of subsections.
    Uses guided JSON decoding to guarantee valid schema-conformant output.

    Returns list of {key, summary, evidence, section, subsection_name} dicts.
    """
    if not questions:
        return []

    prompt = build_batch_prompt(batch, questions)
    result = call_llm_json(
        client, CHUNK_SYSTEM_PROMPT, prompt,
        max_tokens=CHUNK_MAX_TOKENS,
        response_model=ChunkQAResponse,
    )

    if result is None:
        log.warning(f"    LLM returned None for batch of {len(batch)} subsections")
        return []

    answers = result.get("answers", [])
    if not answers:
        return []

    # Build lookup for subsection metadata
    sub_lookup = {sub["name"]: sub for sub in batch}

    found_items = []
    for item in answers:
        if not item.get("found") or not item.get("key") or not item.get("summary"):
            continue

        # Match to source subsection
        sub_name = item.get("subsection", "")
        source_sub = sub_lookup.get(sub_name)

        if source_sub is None:
            # Fuzzy match: check for partial name overlap
            for name, sub in sub_lookup.items():
                if name.lower() in sub_name.lower() or sub_name.lower() in name.lower():
                    source_sub = sub
                    sub_name = name
                    break

        if source_sub is None:
            # Fallback to first subsection in batch
            source_sub = batch[0]
            sub_name = batch[0]["name"]

        found_items.append({
            "key": item["key"],
            "summary": item["summary"],
            "evidence": item.get("evidence", ""),
            "section": source_sub["section"],
            "subsection_name": sub_name,
        })

    return found_items


print("Batch chunk Q&A functions loaded (guided JSON via Pydantic).")

## Step 5: Synthesis (Phase 2)

When the same factor key appears in multiple subsections, the LLM consolidates them into a single entry with a unified summary and the 2-4 most informative evidence quotes.

Single-entry factors skip the LLM call entirely (no synthesis needed).

In [ ]:
SYNTHESIS_SYSTEM_PROMPT = """\
You are a financial analyst consolidating information about a single thematic factor from multiple subsections of an SEC filing. Respond with valid JSON matching the required schema."""


def synthesize_factor(
    client: OpenAI, key: str, category: str, entries: list[dict]
) -> dict:
    """
    Synthesize multiple chunk-level answers for the same factor into one entry.
    If only one entry exists, use it directly (no LLM call needed).
    Uses guided JSON decoding for multi-entry synthesis.
    """
    if len(entries) == 1:
        e = entries[0]
        return {
            "key": key,
            "category": category,
            "summary": e["summary"],
            "evidence": [
                {
                    "text": e["evidence"],
                    "section": e["section"],
                    "subsection": e["subsection_name"],
                }
            ],
            "sentiment": None,
        }

    # Multiple entries -- call LLM to synthesize
    input_block = "\n\n".join(
        f"### From subsection: {e['subsection_name']} ({e['section']})\n"
        f"Summary: {e['summary']}\n"
        f'Evidence: \"{e["evidence"]}\"'
        for e in entries
    )

    user_prompt = f"""Factor: {key}

The following summaries and evidence quotes come from different subsections of the same SEC filing, all addressing the same factor.

{input_block}

Consolidate into a single entry with:
- "summary": 2-4 sentence consolidated summary (be directional: positive, negative, or mixed)
- "evidence": array of 2-4 most informative verbatim quotes (do not combine or edit them)

If inputs conflict, note the tension in the summary."""

    result = call_llm_json(
        client, SYNTHESIS_SYSTEM_PROMPT, user_prompt,
        max_tokens=SYNTHESIS_MAX_TOKENS,
        response_model=SynthesisResponse,
    )

    if result and isinstance(result, dict):
        # Build evidence list with source info
        evidence_list = []
        raw_evidence = result.get("evidence", [])
        if isinstance(raw_evidence, list):
            for i, quote in enumerate(raw_evidence):
                if isinstance(quote, str) and quote.strip():
                    source_entry = entries[min(i, len(entries) - 1)]
                    evidence_list.append({
                        "text": quote.strip(),
                        "section": source_entry["section"],
                        "subsection": source_entry["subsection_name"],
                    })

        return {
            "key": key,
            "category": category,
            "summary": result.get("summary", entries[0]["summary"]),
            "evidence": evidence_list if evidence_list else [
                {"text": entries[0]["evidence"], "section": entries[0]["section"],
                 "subsection": entries[0]["subsection_name"]}
            ],
            "sentiment": None,
        }

    # Fallback: if synthesis LLM call failed, use the longest entry
    best = max(entries, key=lambda e: len(e["summary"]))
    return {
        "key": key,
        "category": category,
        "summary": best["summary"],
        "evidence": [
            {"text": e["evidence"], "section": e["section"],
             "subsection": e["subsection_name"]}
            for e in entries[:3]
        ],
        "sentiment": None,
    }


print("Synthesis functions loaded (guided JSON via Pydantic).")

## Step 6: Filing-Level Processing

Orchestrates the full pipeline for one filing:
1. Load subsection JSON, pre-route questions per subsection
2. **Batch** subsections by token count (~6000 tokens/batch)
3. **Phase 1**: concurrent chunk Q&A on all batches (ThreadPoolExecutor)
4. **Phase 2**: single-entry factors skip LLM; multi-entry factors synthesized concurrently

In [ ]:
def process_filing(subsections_file: Path, client: OpenAI) -> dict | None:
    """
    Process one filing: batched concurrent chunk Q&A + concurrent synthesis.
    Returns the factor JSON dict or None on failure.
    """
    with open(subsections_file) as f:
        data = json.load(f)

    ticker = data["ticker"]
    form_type = data["form"]
    filing_date = data["filing_date"]
    subsections = data["subsections"]

    if not subsections:
        log.warning(f"  {ticker} {filing_date}: no subsections")
        return None

    # Get applicable questions for this ticker (based on sub-sector)
    all_questions = get_applicable_questions(ticker)

    if not all_questions:
        log.warning(f"  {ticker}: no applicable questions found")
        return None

    # Pre-route questions for each subsection
    routed_per_sub: list[list[dict]] = []
    for sub in subsections:
        routed = route_questions(sub["name"], sub["text"], all_questions)
        routed_per_sub.append(routed)

    # Batch subsections by token count
    batches = batch_subsections(subsections)

    # For each batch, compute union of routed questions
    batch_questions: list[list[dict]] = []
    sub_idx = 0
    for batch in batches:
        seen_keys: set[str] = set()
        union_questions: list[dict] = []
        for _ in batch:
            for q in routed_per_sub[sub_idx]:
                if q["key"] not in seen_keys:
                    seen_keys.add(q["key"])
                    union_questions.append(q)
            sub_idx += 1
        batch_questions.append(union_questions)

    log.info(
        f"  {ticker} {form_type} {filing_date}: "
        f"{len(subsections)} subsections -> {len(batches)} batches"
    )

    # Phase 1: Concurrent chunk-level Q&A on batches
    all_chunk_answers: list[dict] = []

    with ThreadPoolExecutor(max_workers=MAX_IN_FLIGHT_LLM) as pool:
        futures = {
            pool.submit(process_batch, client, batch, questions): i
            for i, (batch, questions) in enumerate(zip(batches, batch_questions))
        }
        for future in as_completed(futures):
            batch_idx = futures[future]
            try:
                answers = future.result()
                all_chunk_answers.extend(answers)
            except Exception as e:
                log.warning(f"    Batch {batch_idx} failed: {e}")

    log.info(
        f"  {ticker} {form_type} {filing_date}: "
        f"{len(all_chunk_answers)} chunk-level answers from {len(batches)} batches"
    )

    if not all_chunk_answers:
        log.warning(f"  {ticker} {filing_date}: no factors extracted")
        return None

    # Phase 2: Group by factor key and synthesize
    grouped: dict[str, list[dict]] = defaultdict(list)
    for answer in all_chunk_answers:
        grouped[answer["key"]].append(answer)

    # Build category lookup from all_questions
    key_to_category: dict[str, str] = {}
    for category_name, questions in all_questions.items():
        for key in questions:
            key_to_category[key] = category_name

    # Single-entry factors: no LLM call needed
    factors: list[dict] = []
    multi_entry_keys: list[tuple[str, list[dict]]] = []

    for key, entries in grouped.items():
        category = key_to_category.get(key, "unknown")
        if len(entries) == 1:
            factors.append(synthesize_factor(client, key, category, entries))
        else:
            multi_entry_keys.append((key, entries))

    # Multi-entry factors: concurrent synthesis
    if multi_entry_keys:
        with ThreadPoolExecutor(max_workers=MAX_IN_FLIGHT_LLM) as pool:
            futures = {
                pool.submit(
                    synthesize_factor, client, key,
                    key_to_category.get(key, "unknown"), entries
                ): key
                for key, entries in multi_entry_keys
            }
            for future in as_completed(futures):
                key = futures[future]
                try:
                    factors.append(future.result())
                except Exception as e:
                    log.warning(f"    Synthesis failed for {key}: {e}")

    log.info(f"  {ticker} {form_type} {filing_date}: {len(factors)} factors after synthesis")

    return {
        "ticker": ticker,
        "form": form_type,
        "filing_date": filing_date,
        "model": MODEL,
        "num_factors": len(factors),
        "factors": factors,
    }


print("Filing-level processing function loaded.")

## Step 7: Filing Discovery

Load tickers, discover subsection files, compute output paths, and check resume state.
Prints a dry-run summary before actual processing.

In [ ]:
# Load tickers
with open(TICKER_MAPPING_PATH) as f:
    ticker_mapping = json.load(f)

all_tickers = sorted(
    t for sector_list in ticker_mapping.values() for t in sector_list
)
print(f"Loaded {len(all_tickers)} tickers across {len(ticker_mapping)} sub-sectors")


# ── Dry-run summary ───────────────────────────────────────────────────────
total_files = 0
total_already = 0
total_remaining = 0
tickers_with_data = 0
tickers_no_data = []

for ticker in all_tickers:
    ticker_input = SUBSECTIONS_DIR / ticker
    if not ticker_input.exists():
        tickers_no_data.append(ticker)
        continue

    sub_files = sorted(ticker_input.glob("*_subsections.json"))
    if not sub_files:
        tickers_no_data.append(ticker)
        continue

    tickers_with_data += 1
    ticker_output = FACTORS_DIR / ticker

    for sf in sub_files:
        total_files += 1
        stem = sf.stem.replace("_subsections", "")
        out_file = ticker_output / f"{stem}_factors.json"
        if out_file.exists():
            total_already += 1
        else:
            total_remaining += 1

print(f"\n{tickers_with_data}/{len(all_tickers)} tickers have subsection files")
print(f"{total_files} total subsection files")
print(f"{total_already} already processed (will be skipped)")
print(f"{total_remaining} remaining to process")

if tickers_no_data:
    print(f"\nTickers with no subsection data ({len(tickers_no_data)}): {', '.join(tickers_no_data)}")

## Step 8: Run Factor Extraction

Main processing loop. For each ticker and filing:
1. Skip if output already exists (resume-safe)
2. Call `process_filing()` for chunk Q&A + synthesis
3. Write factor JSON to disk

Progress is tracked via `tqdm` with live counters.

In [ ]:
# Build flat list of (ticker, subsection_file) pairs for progress tracking
all_jobs: list[tuple[str, Path]] = []
for ticker in all_tickers:
    ticker_input = SUBSECTIONS_DIR / ticker
    if not ticker_input.exists():
        continue
    for sf in sorted(ticker_input.glob("*_subsections.json")):
        all_jobs.append((ticker, sf))

print(f"{len(all_jobs)} filing jobs queued")

g_ok = g_skip = g_err = g_factors = 0

pbar = tqdm(all_jobs, desc="Factor extraction", unit="filing")
for ticker, sub_file in pbar:
    ticker_output = FACTORS_DIR / ticker
    ticker_output.mkdir(parents=True, exist_ok=True)

    stem = sub_file.stem.replace("_subsections", "")
    out_file = ticker_output / f"{stem}_factors.json"

    # Resume-safe
    if out_file.exists():
        g_skip += 1
        pbar.set_postfix(ok=g_ok, skip=g_skip, err=g_err, factors=g_factors)
        continue

    try:
        result = process_filing(sub_file, client)
        if result:
            with open(out_file, "w") as f:
                json.dump(result, f, indent=2)
            g_ok += 1
            g_factors += result["num_factors"]
        else:
            g_err += 1
    except Exception as e:
        g_err += 1
        log.error(f"  {sub_file.name}: CRASHED -- {e}")

    pbar.set_postfix(ok=g_ok, skip=g_skip, err=g_err, factors=g_factors)

print(f"\nDone. {g_ok} processed, {g_skip} skipped, {g_err} errors")
print(f"Total factors extracted: {g_factors}")
if g_ok > 0:
    print(f"Avg factors/filing: {g_factors / g_ok:.1f}")

## Step 9: Extraction Summary

Aggregate stats across all tickers, display per-ticker factor file counts, and inspect a sample factor JSON.

In [ ]:
# ── Aggregate stats ───────────────────────────────────────────────────────
ticker_stats = []
total_factor_files = 0
total_factors_all = 0
error_tickers = []

for ticker in all_tickers:
    ticker_dir = FACTORS_DIR / ticker
    if not ticker_dir.exists():
        continue

    factor_files = sorted(ticker_dir.glob("*_factors.json"))
    n_files = len(factor_files)
    total_factor_files += n_files

    n_factors = 0
    n_errors = 0
    for ff in factor_files:
        try:
            with open(ff) as f:
                data = json.load(f)
            n_factors += data.get("num_factors", 0)
        except Exception:
            n_errors += 1

    total_factors_all += n_factors
    ticker_stats.append({
        "ticker": ticker, "files": n_files,
        "factors": n_factors, "errors": n_errors,
    })
    if n_errors > 0:
        error_tickers.append(ticker)

print("=" * 60)
print(f"FACTOR EXTRACTION SUMMARY")
print(f"  Total tickers with output: {len(ticker_stats)}")
print(f"  Total factor files: {total_factor_files}")
print(f"  Total factors: {total_factors_all}")
if total_factor_files > 0:
    print(f"  Avg factors/filing: {total_factors_all / total_factor_files:.1f}")

if error_tickers:
    print(f"\nTickers with JSON errors: {', '.join(error_tickers)}")

# ── Per-ticker breakdown ──────────────────────────────────────────────────
print(f"\nPer-ticker factor file counts:")
for s in sorted(ticker_stats, key=lambda x: x["files"], reverse=True)[:20]:
    print(f"  {s['ticker']:6s}: {s['files']:3d} files, {s['factors']:4d} factors")
if len(ticker_stats) > 20:
    print(f"  ... and {len(ticker_stats) - 20} more tickers")

# ── Sample output ─────────────────────────────────────────────────────────
sample_file = None
for ticker in all_tickers:
    ticker_dir = FACTORS_DIR / ticker
    if ticker_dir.exists():
        files = sorted(ticker_dir.glob("*_factors.json"))
        if files:
            sample_file = files[0]
            break

if sample_file:
    with open(sample_file) as f:
        sample = json.load(f)
    print(f"\n{'=' * 60}")
    print(f"SAMPLE: {sample_file.name}")
    print(f"  Ticker: {sample['ticker']}, Form: {sample['form']}, Date: {sample['filing_date']}")
    print(f"  Model: {sample['model']}")
    print(f"  Factors: {sample['num_factors']}")
    for fac in sample["factors"][:3]:
        print(f"\n  [{fac['category']}] {fac['key']}")
        print(f"    Summary: {fac['summary'][:120]}...")
        print(f"    Sentiment: {fac['sentiment']}")
        if fac["evidence"]:
            ev = fac["evidence"][0]
            print(f"    Evidence: {ev['text'][:100]}...")
            print(f"    Source: {ev['section']} / {ev['subsection'][:50]}")
    if sample["num_factors"] > 3:
        print(f"\n  ... and {sample['num_factors'] - 3} more factors")
else:
    print("\nNo factor files found yet. Run Step 8 first.")